# основные операторы

## 1.1. CREATE TABLE 

CREATE TABLE customers (

    customer_id   INT PRIMARY KEY,
    
    customer_name  VARCHAR(100) NOT NULL,
    
    city           VARCHAR(100),
    
    signup_date    DATE NOT NULL
    
);

CREATE TABLE categories (

    category_id    INT PRIMARY KEY,
    
    category_name  VARCHAR(100) NOT NULL
    
);

CREATE TABLE products (

    product_id     INT PRIMARY KEY,
    
    product_name   VARCHAR(150) NOT NULL,
    
    category_id    INT REFERENCES categories(category_id),
    
    price          DECIMAL(10, 2) NOT NULL
    
);

CREATE TABLE orders (

    order_id       INT PRIMARY KEY,
    
    customer_id    INT REFERENCES customers(customer_id),
    
    order_date     DATE NOT NULL,
    
    status         VARCHAR(30) NOT NULL,
    
    discount       DECIMAL(5, 2) DEFAULT 0
    
);

CREATE TABLE order_items (

    order_id       INT REFERENCES orders(order_id),
    
    product_id     INT REFERENCES products(product_id),
    
    quantity       INT NOT NULL,
    
    unit_price     DECIMAL(10, 2) NOT NULL,
    
    PRIMARY KEY (order_id, product_id)
    
);


## 1.2. **Select** 

SELECT CURRENT_DATE;

SELECT COUNT(*) AS customers_count FROM customers;


**В общем виде**

SELECT ...

FROM ... / JOIN ...

WHERE ...

GROUP BY ... # SELECT категория, SUM(цена) AS total_price FROM товары GROUP BY категория

HAVING ... # в HAVING можно наложить условия на результаты группировки

ORDER BY ... DESC/ASC

LIMIT ...;

**агрегирующие функции**

COUNT() — считает число строк в группе.SUM() — считает сумму чисел в столбце.AVG() — считает среднее значение.MIN() / MAX()

игнорируют NULL

**AS + CASE + distinct**

AS — алиас столбца или таблицы;

CASE — условная логика;

DISTINCT — удаление дублей в проекции, но не исправление ошибок JOIN

SELECT
    product_name,
    price,
    price * 1.20 AS price_with_tax,
    CASE
        WHEN price < 1000 THEN 'budget'
        WHEN price < 5000 THEN 'standard'
        ELSE 'premium'
    END AS price_segment
FROM products;

**NULL**

COALESCE(value, replacement) — замена NULL;

NULLIF(a, b) — превращает a в NULL, если a = b;

NULL нельзя проверять через = NULL или <> NULL; используются IS NULL и IS NOT NULL

# join

In [ ]:
SELECT

    o.order_id,

    o.order_date,

    c.customer_name,

    p.product_name,

    oi.quantity,

    oi.quantity * oi.unit_price AS line_amount

FROM orders o

JOIN customers c ON c.customer_id = o.customer_id

JOIN order_items oi ON oi.order_id = o.order_id

JOIN products p ON p.product_id = oi.product_id;

# оконные функции

function(...) OVER ( #задаёт «окно» 

    PARTITION BY ... #делит это окно

    ORDER BY ... #совсем другая история: он задаёт порядок, в котором функция будет обходить строки раздела.
    
    ROWS BETWEEN ...
)

Оконная функция в SQL состоит из пяти ключевых элементов, каждый из которых строго определяет её поведение.

1. function - Сама функция (например, SUM, ROW_NUMBER, LAG) задаёт тип вычисления — агрегатное, ранжирующее или смещения, и применяется не к группе в целом, а к динамическому набору строк, связанному с текущей записью.

2. Ключевое слово OVER() является обязательным маркером оконной конструкции и отделяет вычисления от обычных скалярных операций, при этом пустые скобки означают окно из всех строк результирующего набора.

3. Предложение PARTITION BY разбивает строки на независимые логические группы (партиции), внутри которых функция перезапускается, — аналогично GROUP BY, но без свёртки, так что каждая строка сохраняется, а расчёт ведётся в пределах своей партиции.

4. Предложение ORDER BY внутри OVER устанавливает порядок строк внутри партиции (или всего набора), что критически важно для ранжирующих функций, а для агрегатов включает режим накопительного итога по умолчанию.

5. Опциональная рамка (ROWS | RANGE BETWEEN ...), задаваемая после ORDER BY, явно ограничивает множество соседних строк, участвующих в вычислении для текущей строки, позволяя реализовать скользящие средние, кумулятивные суммы с границами или сравнения с фиксированным смещением.

нумеруем сотрудников по убыванию зарплаты, причём отдельно для каждого отдела:

SELECT

    name,
    
    department,
    
    salary,
    
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank

FROM employees;

Накопительный итог

SELECT 

    date,
    
    sales,
    
    SUM(sales) OVER (ORDER BY date) AS total
    
    AVG(sales) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS 3_day_av
    
FROM daily_sales    

# анализ и отладка запросов